# 00 — Scrape WikiArt: tambah shard dataset

Menambah data training dengan meng-unduh **shard parquet** tambahan dari
`huggan/wikiart` (HF Hub), meng-ekstrak gambar + label, lalu meng-append ke
`ml/data/raw/`.

**Kenapa:** subset sekarang 1.132 gambar (1 shard) -> model overfit berat
(train f1 ~0.95 vs val ~0.65). Target ~5.000+ gambar (4 shard lagi).

**Strategi** (sesuai CLAUDE.md): `hf_hub_download` per file parquet
(~450-500 MB/shard), BUKAN `load_dataset` (mencoba unduh semua 72 shard / ~37 GB).

**Lisensi:** WikiArt = riset non-komersial. Jangan commit gambar ke git
(sudah di-`.gitignore`), jangan redistribusi.

> Idempoten: shard yang sudah diproses dicatat di `data/raw/_shards_processed.json`
> dan dilewati bila notebook dijalankan ulang. Dedup gambar via hash MD5.

## 1. Konfigurasi

In [ ]:
import sys
from pathlib import Path

_p = Path.cwd()
while not ((_p / "src").is_dir() and (_p / "configs").is_dir()):
    if _p == _p.parent:
        raise RuntimeError("folder ml/ tidak ketemu dari " + str(Path.cwd()))
    _p = _p.parent
if str(_p) not in sys.path:
    sys.path.insert(0, str(_p))

REPO_ID        = "huggan/wikiart"
SHARDS_TO_ADD  = [1, 2, 3, 4]      # shard 0 sudah ada. Tiap shard ~1.100-1.200 gambar
PARQUET_TMPL   = "data/train-{i:05d}-of-00072.parquet"
MIN_RESOLUTION = 100              # buang gambar < 100 px (sisi terpendek)
JPEG_QUALITY   = 95
DELETE_PARQUET_AFTER = True       # hapus parquet dari cache HF setelah ekstrak (hemat disk)

RAW      = _p / "data" / "raw"
MANIFEST = RAW / "_shards_processed.json"
META_CSV = RAW / "metadata.csv"
print("data/raw :", RAW)
print("shard target:", SHARDS_TO_ADD, "-> perkiraan +", len(SHARDS_TO_ADD) * 1150, "gambar")

## 2. Nama label (artist / genre / style) dari `dataset_infos.json`

In [ ]:
import json
from huggingface_hub import hf_hub_download

_info_path = hf_hub_download(REPO_ID, "dataset_infos.json", repo_type="dataset")
_feats = json.load(open(_info_path, encoding="utf-8"))["huggan--wikiart"]["features"]
artist_names = _feats["artist"]["names"]   # 129
genre_names  = _feats["genre"]["names"]    # 11
style_names  = _feats["style"]["names"]    # 27
print(f"artist {len(artist_names)} | genre {len(genre_names)} | style {len(style_names)} kelas")
print("style:", style_names)

## 3. State: manifest, metadata lama, index file berikutnya, hash gambar lama

In [ ]:
import hashlib

done_shards = json.loads(MANIFEST.read_text()) if MANIFEST.exists() else ([0] if META_CSV.exists() else [])
print("shard sudah diproses:", done_shards)

import pandas as pd
old_meta = pd.read_csv(META_CSV) if META_CSV.exists() else pd.DataFrame(
    columns=["filename", "artist", "genre", "style", "artist_name", "genre_name", "style_name"])
print("metadata.csv:", len(old_meta), "baris")

existing = sorted(RAW.glob("wikiart_*.jpg"))
next_idx = 1 + max((int(f.stem.split("_")[1]) for f in existing), default=-1)
print("file gambar ada:", len(existing), "| index berikutnya: wikiart_%05d.jpg" % next_idx)

print("menghitung hash gambar lama (dedup)...")
seen_hashes = {hashlib.md5(f.read_bytes()).hexdigest() for f in existing}
print("hash unik:", len(seen_hashes))

## 4. Unduh + ekstrak tiap shard

In [ ]:
import io, os
from PIL import Image

new_rows = []
for shard in SHARDS_TO_ADD:
    if shard in done_shards:
        print(f"shard {shard}: sudah diproses, lewati")
        continue
    fn = PARQUET_TMPL.format(i=shard)
    print(f"\nshard {shard}: unduh {fn} ...")
    pq_path = hf_hub_download(REPO_ID, fn, repo_type="dataset")
    df = pd.read_parquet(pq_path)          # ~500 MB, seluruh shard ke RAM
    if shard == SHARDS_TO_ADD[0]:
        _f0 = df.iloc[0]["image"]
        print(f"  kolom: {list(df.columns)}")
        print(f"  image[0] tipe: {type(_f0).__name__}"
              + (f", keys {list(_f0.keys())}" if isinstance(_f0, dict) else ""))
    print(f"  {len(df)} baris")

    kept = corrupt = lowres = dup = 0
    for _, r in df.iterrows():
        fld = r["image"]
        try:
            if isinstance(fld, Image.Image):
                im = fld.convert("RGB")
                data = im.tobytes()
            else:
                data = fld["bytes"] if isinstance(fld, dict) else fld
                Image.open(io.BytesIO(data)).verify()
                im = Image.open(io.BytesIO(data)).convert("RGB")
        except Exception:
            corrupt += 1
            continue
        h = hashlib.md5(data).hexdigest()
        if h in seen_hashes:
            dup += 1
            continue
        if min(im.size) < MIN_RESOLUTION:
            lowres += 1
            continue

        seen_hashes.add(h)
        fname = f"wikiart_{next_idx:05d}.jpg"
        im.save(RAW / fname, "JPEG", quality=JPEG_QUALITY)
        a, g, s = int(r["artist"]), int(r["genre"]), int(r["style"])
        new_rows.append({
            "filename": fname, "artist": a, "genre": g, "style": s,
            "artist_name": artist_names[a], "genre_name": genre_names[g],
            "style_name": style_names[s],
        })
        next_idx += 1
        kept += 1

    print(f"  +{kept} disimpan | korup {corrupt} | <{MIN_RESOLUTION}px {lowres} | duplikat {dup}")
    done_shards.append(shard)
    MANIFEST.write_text(json.dumps(sorted(done_shards)))
    if DELETE_PARQUET_AFTER:
        try:
            os.remove(pq_path)
            print("  parquet dihapus dari cache")
        except OSError:
            pass

print(f"\n=== TOTAL gambar baru: {len(new_rows)} ===")

## 5. Gabung ke `metadata.csv`

In [ ]:
merged = pd.concat([old_meta, pd.DataFrame(new_rows)], ignore_index=True)
merged = merged.drop_duplicates("filename").reset_index(drop=True)
merged.to_csv(META_CSV, index=False)
print(f"metadata.csv: {len(old_meta)} -> {len(merged)} baris")
print(f"file .jpg di data/raw: {len(list(RAW.glob('wikiart_*.jpg')))}")

## 6. Distribusi kelas: sebelum vs sesudah

In [ ]:
import matplotlib.pyplot as plt

b = old_meta["style_name"].value_counts()
a = merged["style_name"].value_counts()
allc = sorted(set(b.index) | set(a.index), key=lambda k: -a.get(k, 0))
x = range(len(allc))

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar([i - 0.2 for i in x], [b.get(k, 0) for k in allc], width=0.4, label=f"sebelum ({len(old_meta)})")
ax.bar([i + 0.2 for i in x], [a.get(k, 0) for k in allc], width=0.4, label=f"sesudah ({len(merged)})")
ax.set_xticks(list(x)); ax.set_xticklabels(allc, rotation=90)
ax.set_ylabel("jumlah gambar"); ax.legend()
ax.set_title(f"Distribusi style: {b.shape[0]} -> {a.shape[0]} kelas | "
             f"imbalance {a.max()/a.min():.0f}x (sebelum {b.max()/b.min():.0f}x)")
fig.tight_layout(); fig.savefig(_p / "outputs" / "scrape_distribusi.png", dpi=110); plt.show()

## 7. WAJIB dijalankan setelah scraping

Data mentah bertambah, tapi turunannya masih lama:

1. **Regenerate split** — jalankan sel 8 di bawah (atau `eda_wikiart.ipynb` Section 8).
2. **Hapus cache label** — `ml/data/cache/label_to_idx__*.json` (auto-regen saat training).
3. **Preprocess ulang** — dari folder `ml/`:
   ```
   python -m src.preprocess --config configs/config.yaml
   ```
   (gambar baru belum di-SquarePad; `overwrite: false` -> yang lama dilewati, hanya yang baru diproses)
4. **Retrain** — `01_train.ipynb` atau `python -m src.train`.
5. **Checkpoint lama** (`checkpoints/best.pth`) tidak valid lagi (jumlah kelas bisa berubah).

## 8. Regenerate stratified split (train/val/test)

In [ ]:
from sklearn.model_selection import train_test_split

MIN_SAMPLES = 40     # 5.6k gambar -> ambil 11 kelas solid (Cubism 133 s/d Impressionism 1660);
                     # buang 5 kelas mungil (Pointillism/Fauvism/dll <=16 sampel)
sc = merged["style_name"].value_counts()
valid = sc[sc >= MIN_SAMPLES].index
f = merged[merged["style_name"].isin(valid)].copy()
print(f"sebelum filter: {len(merged)} gambar / {merged['style_name'].nunique()} kelas")
print(f"sesudah filter: {len(f)} gambar / {f['style_name'].nunique()} kelas")

train_df, temp = train_test_split(f, test_size=0.3, stratify=f["style_name"], random_state=42)
val_df, test_df = train_test_split(temp, test_size=0.5, stratify=temp["style_name"], random_state=42)
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    d.to_csv(RAW / f"{name}_split.csv", index=False)
    print(f"{name}_split.csv: {len(d)}")

# hapus cache label supaya training regenerate sesuai kelas baru
for c in (_p / "data" / "cache").glob("label_to_idx__*.json"):
    c.unlink(); print("hapus cache:", c.name)